# Timestral in TabTune

**Timestral** (Mistral AI) is a family of time series foundation models for multivariate
probabilistic forecasting. It is a causal patch transformer trained with **Decoupled
Horizon Forecasting**: separate prediction tokens read the causally encoded context, and a
monotone head emits **21 non-crossing quantiles** per step. Every variate is a target, a
past-only covariate or a known-future covariate. Covariates condition the targets and
never read them.

Four checkpoints: **5M, 28M, 123M and 209M** parameters (209M is the default), with a
context of up to **32,768 steps**. TabTune vendors the `timestral` 1.0.0 package, so
nothing beyond `tabtune` is installed.

---

<center>

<div style="text-align:center;">
<a href="https://tabtune.lexsi.ai/"><img src="https://raw.githubusercontent.com/Lexsi-Labs/TabTune/refs/heads/docs/assets/tabtunelogo.png" width="450"></a>

<a href="https://tabtune.lexsi.ai/"><img src="https://raw.githubusercontent.com/Lexsi-Labs/TabTune/refs/heads/docs/assets/docs.png" width="150"></a>
<a href="https://discord.gg/dSB62Q7A"><img src="https://raw.githubusercontent.com/Lexsi-Labs/TabTune/refs/heads/docs/assets/discord.png" width="150"></a>
<a href="https://arxiv.org/abs/2511.02802"><img src="https://raw.githubusercontent.com/Lexsi-Labs/TabTune/refs/heads/docs/assets/arxiv.png" width="150"></a>

<a href="https://lexsi.ai/"><img src="https://raw.githubusercontent.com/Lexsi-Labs/TabTune/refs/heads/docs/assets/lexsilogo.png" width="250"></a>
</div>

Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/Lexsi-Labs/TabTune">Github</a> </i> ⭐

To run this, press "*Runtime*" and press "*Run all*" on a Google Colab instance!
</center>

---

## 0. Setup

In [ ]:
# On Colab, install TabTune first:
# !pip install -q tabtune
import logging
import os

import numpy as np
import pandas as pd

import tabtune
from tabtune.logger import setup_logger

setup_logger(level=logging.WARNING)  # INFO shows every fit, backtest and training report
print("TabTune", tabtune.__version__)

### Which checkpoint

`mistralai/Timestral-28M` is the default here: small enough for a free Colab GPU or a
laptop CPU, and it downloads from the Hugging Face Hub on first use (no token). Set
`TIMESTRAL_CHECKPOINT` to another Hub id or a local directory to switch.

`TIMESTRAL_CHECKPOINT=random` builds a tiny randomly initialised Timestral instead. That
runs with no network and checks the plumbing. Its forecasts mean nothing.

In [ ]:
CHECKPOINT = os.environ.get("TIMESTRAL_CHECKPOINT", "mistralai/Timestral-28M")


def tiny_random_checkpoint(path="timestral-random"):
    """A 16-wide random Timestral saved as config.json + model.safetensors."""
    import torch
    from safetensors.torch import save_file

    from tabtune.models.timestral import TimestralConfig, TimestralModel

    knots = (0.01, 0.05) + tuple(round(0.1 * i, 1) for i in range(1, 10)) + (0.95, 0.99)
    config = TimestralConfig(
        patch_size=4, d_model=16, num_heads=2, mlp_hidden_dim=32,
        block_schedule=("time_variate", "variate_time", "time"), quantile_knots=knots,
        block_size=8, context_length=512,
    )
    torch.manual_seed(0)
    os.makedirs(path, exist_ok=True)
    config.save(os.path.join(path, "config.json"))
    save_file(TimestralModel(config).state_dict(), os.path.join(path, "model.safetensors"))
    return path


if CHECKPOINT == "random":
    CHECKPOINT = tiny_random_checkpoint()

MODEL_PARAMS = {"checkpoint": CHECKPOINT}
print("checkpoint:", CHECKPOINT)

### Synthetic data

Synthetic, so the only download is the checkpoint. Six hourly series with a daily season,
two weeks long. The last 24 hours are held out.

In [ ]:
from tabtune.TimeSeries import TimeSeriesPipeline, TimeSeriesSchema, make_panel, split_horizon

H = 24
df = make_panel(n_series=6, length=24 * 14, freq="h", seed=0)
schema = TimeSeriesSchema(target="target", item_id="item_id")
history, actual = split_horizon(df, schema, prediction_length=H)

print(history.shape, actual.shape)
history.head()

---

## 1. The registry entry

No weights, no network. The registry answers most planning questions before anything
loads.

In [ ]:
from tabtune.registry import get_time_series_model_spec

spec = get_time_series_model_spec("Timestral")

print(f"name              {spec.name}")
print(f"tasks             {sorted(spec.tasks)}")
print(f"strategies        {sorted(spec.strategies)}")
print(f"checkpoints       {', '.join(spec.checkpoints)}")
print(f"default           {spec.default_checkpoint}")
print(f"max context       {spec.max_context:,}")
print(f"multivariate      {spec.supports_multivariate}")
print(f"covariates        {spec.supports_covariates} (categorical: {spec.supports_categorical_covariates})")
print(f"NaN in history    {spec.native_missing}")
print(f"licence           {spec.license.name}")
print(f"commercial        {spec.license.commercial_use_ok}  (None = unverified)")
print(f"alternatives      {spec.commercial_alternatives}")

### Name resolution

`Timestral-209M` and `Mistral-Timestral` are aliases. Case, hyphens and underscores are
ignored.

In [ ]:
for alias in ["Timestral", "timestral", "Timestral-209M", "mistral_timestral"]:
    print(f"  {alias:<18} -> {get_time_series_model_spec(alias).name}")

---

## 2. Limits

| Limit | Value | What happens when you exceed it |
|---|---:|---|
| Context | `32,768` steps | Truncated to the most recent steps (`envelope_mode="warn"`), or an error (`"error"`) |
| Horizon | none | Decoded block by block, feeding back the median |
| Quantiles | 0.01 to 0.99 | Levels outside the native grid are rejected |
| Missing values | allowed | Read as missing by the model; shorter series are left-padded with NaN |

The weight licence is Mistral AI's Modified MIT. The registry records it as unverified, so
`license_mode="commercial"` warns and leaves the decision to you.

In [ ]:
import warnings

from tabtune.registry import check_license

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    check_license(spec, mode="commercial")
for warning in caught:
    print(f"  [warn] {warning.message}")

---

## 3. Zero-shot forecasting

`fit` validates the history and loads the weights. No training happens for
`tuning_strategy="inference"`.

In [ ]:
pipe = TimeSeriesPipeline(
    "Timestral",
    model_params=MODEL_PARAMS,
    forecast_params={"prediction_length": H, "quantile_levels": [0.1, 0.5, 0.9]},
)
pipe.fit(history, schema)
forecast = pipe.predict()
print("point", forecast.point.shape, " quantiles", forecast.quantiles.shape, "(series, steps, levels)")
forecast.to_pandas().head()

In [ ]:
scores = pipe.evaluate(actual, forecast=forecast)
{k: v for k, v in scores.items() if not isinstance(v, (dict, list, pd.DataFrame))}

### Backtest and conformal calibration

`backtest` scores rolling origins at the end of the history without refitting.
`calibrate` uses the same windows to widen or narrow the quantiles so their coverage
matches the levels you asked for.

In [ ]:
pipe.backtest(windows=3)

In [ ]:
pipe.calibrate(windows=3)
calibrated = pipe.predict()
calibrated_scores = pipe.evaluate(actual, forecast=calibrated)
{k: v for k, v in calibrated_scores.items() if "coverage" in k}

---

## 4. Covariates and several targets

Timestral reads past-only covariates (seen in the history only), known-future covariates
(also given over the horizon in `future_df`) and categorical covariates. Several targets
of one item are forecast jointly.

In [ ]:
cov = make_panel(n_series=4, length=24 * 10 + H, freq="h", seed=1, covariates=("temp", "promo"))
cov["store_type"] = np.where(cov.groupby("item_id").cumcount() % 3 == 0, "mall", "street")
cut = cov.groupby("item_id").cumcount() < 24 * 10
cov_history = cov[cut].reset_index(drop=True)
cov_future = cov[~cut].drop(columns="target").reset_index(drop=True)

cov_schema = TimeSeriesSchema(
    target="target",
    item_id="item_id",
    past_covariates=["temp"],
    known_covariates=["promo", "store_type"],
)
cov_pipe = TimeSeriesPipeline("Timestral", model_params=MODEL_PARAMS, forecast_params={"prediction_length": H})
cov_forecast = cov_pipe.fit(cov_history, cov_schema, future_df=cov_future).predict(future_df=cov_future)
print("point", cov_forecast.point.shape)
cov_forecast.to_pandas().head()

In [ ]:
multi = make_panel(n_series=3, length=24 * 10, freq="h", seed=2, n_targets=2)
multi_schema = TimeSeriesSchema(target=["target_0", "target_1"], item_id="item_id")
multi_forecast = (
    TimeSeriesPipeline("Timestral", model_params=MODEL_PARAMS, forecast_params={"prediction_length": H})
    .fit(multi, multi_schema)
    .predict()
)
print(multi_forecast.point.shape, "(one row per item and target)")

---

## 5. Anomalies, gaps and embeddings

The other three tasks run on the same forecaster.

In [ ]:
from tabtune.TimeSeries import make_anomalous_panel

anomalous = make_anomalous_panel(n_series=3, length=24 * 10, n_anomalies=4, seed=0)
labels = anomalous[["item_id", "timestamp", "is_anomaly"]]
detector = TimeSeriesPipeline(
    "Timestral",
    task_type="anomaly_detection",
    model_params=MODEL_PARAMS,
    task_params={"method": "interval", "alpha": 0.01},
)
anomalies = detector.fit(anomalous.drop(columns="is_anomaly"), schema).predict()
print(anomalies)
anomalies.evaluate(labels)

In [ ]:
gappy = df.copy()
gappy.loc[gappy.groupby("item_id").cumcount().between(200, 211), "target"] = np.nan
filled = (
    TimeSeriesPipeline("Timestral", task_type="imputation", model_params=MODEL_PARAMS)
    .fit(gappy, schema)
    .predict()
    .to_pandas()
)
filled[filled["imputed"]].head()

In [ ]:
embeddings = (
    TimeSeriesPipeline("Timestral", task_type="embedding", model_params=MODEL_PARAMS)
    .fit(history, schema)
    .predict()
)
print(embeddings.embeddings.shape, "(one vector per series, d_model wide)")

---

## 6. Fine-tuning and LoRA

`finetune` updates every weight. `peft` trains LoRA adapters on the attention (`wQKV`,
`wO`) and MLP (`mlp.w12`, `mlp.w3`) projections and leaves the rest frozen. Training
windows are sampled from the history. The loss is the pinball loss over the 21 native
quantiles, in the model's normalised space, so it stays finite on large-valued series.
`training_report_` records the run.

In [ ]:
def run_tuning(strategy):
    tuned = TimeSeriesPipeline(
        "Timestral",
        tuning_strategy=strategy,
        model_params=MODEL_PARAMS,
        forecast_params={"prediction_length": H},
        tuning_params={"epochs": 2, "steps_per_epoch": 10, "batch_size": 16, "learning_rate": 1e-4, "seed": 0},
    ).fit(history, schema)
    report = tuned.training_report_
    print(f"{strategy:<9} steps={report['steps_run']}  final loss={report['final_train_loss']:.4f}  ({report['objective']})")
    return tuned


tuned = {strategy: run_tuning(strategy) for strategy in ("finetune", "peft")}

---

## 7. Against the baselines

`TimeSeriesLeaderboard` backtests each candidate on the same windows. With the `random`
checkpoint, expect Timestral to lose to the seasonal baseline.

In [ ]:
from tabtune.TimeSeries import TimeSeriesLeaderboard

board = TimeSeriesLeaderboard(history, schema, forecast_params={"prediction_length": H}, windows=2)
board.add_models(["SeasonalNaive", "Naive"])
board.add_model("Timestral", model_params=MODEL_PARAMS, label="Timestral zero-shot")
board.add_model(
    "Timestral",
    tuning_strategy="peft",
    model_params=MODEL_PARAMS,
    tuning_params={"epochs": 1, "steps_per_epoch": 10, "batch_size": 16, "learning_rate": 1e-4, "seed": 0},
    label="Timestral LoRA",
)
board.run()

---

## 8. Save and load

The fitted pipeline, history and any fine-tuned weights go into one file.

In [ ]:
path = "timestral_peft.joblib"
tuned["peft"].save(path)
restored = TimeSeriesPipeline.load(path)
np.testing.assert_allclose(restored.predict().point, tuned["peft"].predict().point, atol=1e-5)
print("restored forecasts match")

---

## 9. Timestral - Summary

| Scenario | Verdict |
|---|---|
| Zero-shot probabilistic forecasting | Good fit: 21 native quantiles, any horizon |
| Covariates | Good fit: past-only, known-future and categorical |
| Several targets per item | Good fit: forecast jointly |
| Long histories | Good fit: up to 32,768 steps |
| Fine-tuning | `finetune` or `peft` (LoRA on attention and MLP), finite normalised-space loss |
| Anomalies, gaps, embeddings | Supported through the same pipeline |
| Commercial deployment | **Check the licence.** Weights are Modified MIT, recorded as unverified; Chronos-2 is the Apache-2.0 alternative |